# Bayesian Optimisation: ERK Oscillation (v12 — light-budget, AUC-per-light objective)

Same parameter space and FOV-finder setup as v11, but the BO target is a
**real per-cell AUC normalised by total light dose**.  This rewards
patterns that produce the largest sustained ERK response per millisecond
of LED-on time, so the BO is free to drive the dose *down* if a low-budget
pattern is more efficient.

**`auc_per_light` formula (per FOV):**

```
For each cell that passes the quality gates (tracked ≥ 80 % of frames,
baseline_cnr < 1.0):
    baseline_per_cell = mean(CNR_t  for t in baseline frames)
    auc_cell          = Σ (CNR_t - baseline_per_cell)
                            over t ∈ [first_frame_stim, n_frames)
                            (stim + recovery = 80 frames at defaults)
    auc_cell          = max(0, auc_cell)        # clip negative cells to 0

auc_above_baseline  = median over valid cells of auc_cell    (per FOV)
auc_per_light       = auc_above_baseline / (light_budget_ms / 1000)
```

Choices baked into this metric:

- **Window: stim + recovery.**  Captures sustained activation after the
  light turns off, not just the response while light is on.
- **Median over cells.**  Robust to a handful of unusually bright cells
  dominating the score; FOVs of different cell counts compare fairly.
- **Per-cell clip at 0.**  Cells whose net excursion is *below* baseline
  contribute 0, not negative.  A cell oscillating around baseline still
  contributes whatever positive integral it has — only the net-loss
  cells get zeroed.
- **Dose denominator in seconds.**  `light_budget_ms / 1000` so units
  are CNR·frames per second of LED-on time.

| Notebook | BO objective | Search space |
|---|---|---|
| `…_testv11_light_budget.ipynb` | `frac_responders` (fixed dose) | `ramp_fraction × pulse_interval` at fixed `light_budget` |
| `…_testv12_light_budget_auc.ipynb` (this) | **`auc_per_light` (real per-cell AUC)** | `light_budget × ramp_fraction × pulse_interval` |

**Carried over from v11:**

1. `pulse_interval` grid: 1–10 frames step 2 ⇒ {1, 3, 5, 7, 9}.
2. FOV finder pre-screen with `FE_ErkKtr` + CNR < 1.0 in ≥ 75 % of cells.
3. Quality gates unchanged (`max_baseline_cnr=1.0`, `min_track_fraction=0.8`).

Total grid: 57 × 11 × 5 = **3135 candidate conditions**.

> NB: The v11 diagnostic column also called `auc_per_light` still uses
> the old `mean_cnr_above_baseline / light_seconds` formula.  It's only a
> diagnostic there (the BO target is `frac_responders`), so it doesn't
> affect optimisation, but the column is *not* directly comparable across
> the two notebooks.


In [1]:
import os
import time
import logging
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress JAX/XLA debug messages
os.environ["JAX_LOG_COMPILES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax

jax.config.update("jax_log_compiles", False)
for _name in list(logging.Logger.manager.loggerDict):
    if "jax" in _name or "absl" in _name:
        logging.getLogger(_name).setLevel(logging.WARNING)

# ---------------------------------------------------------------------------
# gpax/numpyro compatibility shim
# ---------------------------------------------------------------------------
# numpyro >= 0.20 removed haiku support from numpyro.contrib.module, but
# gpax 0.1.9 eagerly imports viDKL via gpax/models/__init__.py, which tries
# `from numpyro.contrib.module import random_haiku_module, haiku_module`.
# We don't use viDKL (only ExactGP, viGP, viSparseGP), so we install stubs
# that raise only if viDKL is actually called. This MUST run before any
# `import gpax` (including the lazy ones inside the BO agent classes).
import numpyro.contrib.module as _ncm


def _haiku_unavailable(*_args, **_kwargs):
    raise NotImplementedError(
        "haiku support was removed from numpyro >= 0.20. "
        "viDKL is not available. Use ExactGP, viGP, or viSparseGP."
    )


if not hasattr(_ncm, "random_haiku_module"):
    _ncm.random_haiku_module = _haiku_unavailable
if not hasattr(_ncm, "haiku_module"):
    _ncm.haiku_module = _haiku_unavailable
# ---------------------------------------------------------------------------

from faro.core.data_structures import (
    PowerChannel,
    SegmentationMethod,
)
from faro.core.controller import Controller
from faro.core.pipeline import ImageProcessingPipeline
from faro.agents.bo_optimization import (
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)
from faro.agents.bo_oscillation import OscillationBO
import faro.core.utils as utils

## Microscope & pipeline setup

**Microscope:** Jungfrau (no DMD -- full-FOV stimulation).

**Channels:**
- Imaging: miRFP (nuclear marker) + mScarlet3 (ERK-KTR reporter)
- Stimulation: CyanStim (optogenetic activation, power 10)
- Optocheck: mCitrine (verify optoRTK expression, last frame only)

**Pipeline:** CellposeV4 segmentation -> ERK-KTR FE -> Trackpy tracking
-> OptoCheckFE on reference frames. Full-FOV stimulation (no DMD).

In [2]:
from faro.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")
mic.mmc.setProperty("TIPFSStatus", "State", "On")

In [3]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc  # point the widget at our CMMCore instance
viewer.window.add_dock_widget(mm_wdg)  # dock Micro-Manager controls in napari

In [ ]:
WELLS = []
START_COL = 2
END_COL = 7
for i, row in enumerate("ABCDEFGH"):
    cols = (
        range(START_COL, END_COL + 1)
        if i % 2 == 0
        else range(END_COL, START_COL - 1, -1)
    )
    WELLS.extend(f"{row}{c}" for c in cols)

START_COL = 8
END_COL = 11
for i, row in enumerate("ABCDEFGH"):
    cols = (
        range(START_COL, END_COL + 1)
        if i % 2 == 0
        else range(END_COL, START_COL - 1, -1)
    )
    WELLS.extend(f"{row}{c}" for c in cols)

In [ ]:
WELLS

In [ ]:
len(WELLS)

In [ ]:
# --- Experiment parameters ---
TIME_BETWEEN_TIMESTEPS = 60  # seconds (1 frame/min)

# --- Phase layout (frames; 1 frame == 1 minute at TIME_BETWEEN_TIMESTEPS=60) ---
N_FRAMES_BASELINE = 10
N_FRAMES_STIM = 60
N_FRAMES_RECOVERY = 20

# Derived (don't edit -- adjust the three above instead).
N_FRAMES = N_FRAMES_BASELINE + N_FRAMES_STIM + N_FRAMES_RECOVERY  # 90
FIRST_FRAME_STIM = N_FRAMES_BASELINE  # 10
LAST_FRAME_STIM = FIRST_FRAME_STIM + N_FRAMES_STIM  # 70

# --- Plate calibration ---
PLATE_CALIBRATION_PATH = r"./calib_plate_96.json"  # <-- UPDATE

# --- FOV finder knobs ---
FOV_BORDER_UM = 1000.0
FOV_MIN_DISTANCE_UM = 750.0
FOV_MIN_CELLS = 35
FOV_N_CANDIDATES_PER_WELL = 10

# --- Phased FOV layout ---
N_WELLS_PER_PHASE = 6
FOVS_PER_WELL = 3
N_FOVS = N_WELLS_PER_PHASE * FOVS_PER_WELL  # 18 FOVs per phase
N_CONDITIONS_PER_ITER = 3
FOVS_PER_CONDITION = N_FOVS // N_CONDITIONS_PER_ITER
N_PHASES = 13

WELLS = WELLS[: N_PHASES * N_WELLS_PER_PHASE]


# --- Storage ---
base_path = "E:\\Alex"
experiment_name = "2026-05-08_bo_erk_oscillation_v12_light_budget_auc"
path = os.path.join(base_path, experiment_name)

# --- Stimulation channel (base -- exposure is overridden by BO; power fixed) ---
stim_channel = PowerChannel(
    config="CyanStim",
    exposure=100,
    group="TTL_ERK",
    power=10,
)

# --- Imaging channels (acquired every timepoint AND used by FOV finder) ---
imaging_channels = (
    PowerChannel(config="miRFP", exposure=150, group="TTL_ERK", power=95),
    PowerChannel(config="mScarlet3", exposure=150, group="TTL_ERK", power=95),
)

# --- Optocheck channel (acquired on last frame only) ---
optocheck_channel = PowerChannel(
    config="mCitrine",
    exposure=600,
    group="TTL_ERK",
    power=95,
)

In [ ]:
len(WELLS) / N_WELLS_PER_PHASE

In [ ]:
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4

# A single Cellpose instance is shared by the experiment pipeline AND the
# FOVFinderAgent below.  Reusing the same model avoids loading it twice
# (memory + GPU savings) and guarantees that the FOV finder counts cells
# with exactly the same segmentation the experiment will use.
segmentator = CellposeV4(
    custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
    min_size=100,
)

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            name="labels",
            segmentation_class=segmentator,
            use_channel=0,  # segment on miRFP (nuclear marker)
            save_tracked=True,
        )
    ],
    feature_extractor=FE_ErkKtr("labels"),
    tracker=TrackerTrackpy(search_range=50),
    stimulator=StimWholeFOV(),
    feature_extractor_ref=OptoCheckFE(used_mask="labels"),
)

from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)

## FOV selection (per-phase, automated)

Instead of manually selecting 18 positions in napari, a `FOVFinderAgent`
(plugged into the generic `ComposedAgent`) picks fresh positions before
**every phase**:

1. The agent loads the plate calibration (`WellPlatePlan` JSON saved by
   the pymmcore-widgets MDA plate widget).
2. It pops the next `N_WELLS_PER_PHASE` (= 6) wells from `WELLS`.
3. For each well it generates `FOV_N_CANDIDATES_PER_WELL` (= 8) random
   candidate positions, kept `FOV_BORDER_UM` µm away from the well edge
   and at least `FOV_MIN_DISTANCE_UM` apart from each other.
4. It snaps the candidates with the segmentation channel only (miRFP)
   via `mic.run_mda` and segments the result with the **same**
   `CellposeV4` instance the experiment pipeline uses (`segmentator`).
5. Candidates with `< FOV_MIN_CELLS` cells are dropped; the remaining
   ones are reduced to `FOVS_PER_WELL` (= 3) per well by greedy
   farthest-point sampling so the picked FOVs do not overlap.

The result (18 fresh FOVs / phase) is fed straight into
`OscillationBO.run_one_phase`, which is wired up by the `ComposedAgent`
in the next sections.

The segmentator was already created in the pipeline cell above and is
reused here -- no extra setup needed.  The `FOVFinderAgent` is constructed
in the *Configure and run* section together with the BO agent and the
composed driver.


## Oscillation classifier

Load the pre-trained sliding-window oscillation classifier. This model
uses FFT, ACF, and time-domain features extracted from the ERK-KTR
`cnr` trace to classify each window as oscillating or not.

The classifier is restricted at runtime to the **stimulation window
only** (via `OscillationBO.classifier_window`, which defaults to
`(FIRST_FRAME_STIM, LAST_FRAME_STIM)`). Baseline and recovery frames
are acquired but not classified.

The BO target in this notebook is **`frac_responders`** — per FOV, the
fraction of valid cells whose **mean** classifier osc-probability across
the scoring window is `>= frac_responder_threshold` (default 0.75).
This is computed *directly* from the classifier's per-window
probabilities; the legacy 3-gate `frac_oscillating` rule (FFT amplitude
+ max probability + consecutive windows) is **not** used by the BO and
is disabled in the cell below by setting all three thresholds to 0.

Cells are only included in the responder calculation if they pass two
quality gates:
- Tracked for `>= min_track_fraction * n_frames` (default 80 %)
- Baseline CNR `< max_baseline_cnr` (default 1.0)


In [ ]:
import joblib

# --- Load pre-trained oscillation classifier ---
CLASSIFIER_PATH = r"./oscillation_model_60min.joblib"  # <-- UPDATE THIS PATH

model_data = joblib.load(CLASSIFIER_PATH)
osc_clf = model_data["clf"]
osc_scaler = model_data["scaler"]
osc_feature_cols = model_data["feature_cols"]
osc_cfg = model_data["config"]
osc_cfg["window_size"] = model_data["window_size"]
osc_cfg["window_step"] = model_data["window_step"]

# Import the predict_trace function from the classifier script
_classifier_script = "./apply_oscillation_classifier_v2.py"

_spec = importlib.util.spec_from_file_location("osc_classifier", _classifier_script)
_osc_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_osc_module)
predict_trace = _osc_module.predict_trace

# --- Oscillation thresholds ---
# A cell is oscillating iff ALL three gates pass (>= for all):
#   1. FFT amplitude score >= 0.3
#   2. Max osc. probability >= 0.95
#   3. Consecutive osc. windows >= 3
MIN_OSC_PROBABILITY = (
    0.0  # v6: gates disabled, BO target is frac_responders (mean prob >= 0.75 per cell)
)
MIN_CONSECUTIVE_WINDOWS = 0
MIN_FFT_AMPLITUDE = 0.0

print(f"Loaded oscillation classifier from {CLASSIFIER_PATH}")
print(f"  Window: {osc_cfg['window_size']} steps, stride {osc_cfg['window_step']}")
print(
    f"  Thresholds: prob >= {MIN_OSC_PROBABILITY}, "
    f"consecutive >= {MIN_CONSECUTIVE_WINDOWS}, "
    f"fft_amplitude >= {MIN_FFT_AMPLITUDE}"
)

## Batch BO agent

The :class:`OscillationBO` subclass of :class:`BOptGPAX` lives in
``faro.agents.bo_oscillation``.  It overrides:

- ``_create_events_for_batch`` -- builds RTMEvents for ``N_FOVS`` FOVs across
  ``N_CONDITIONS_PER_ITER`` conditions (``FOVS_PER_CONDITION`` each), applying
  per-frame ramped stim exposure.
- ``_preprocess_results`` -- runs the oscillation classifier on each cell's
  ``cnr`` trace (sliced to the stimulation window via ``classifier_window``)
  and computes the per-FOV oscillating fraction. Also computes the
  ``baseline_cnr`` covariate from the first ``n_baseline_frames``.
- ``_on_phase_complete`` -- live plotting + checkpoint saving + saves the
  trained GP to ``{storage_path}/bo_model.joblib`` for post-hoc analysis.

All batch-BO machinery (sequential greedy acquisition with local penalisation,
FOV-index offsetting across phases, ``run_one_phase`` integration with
``ComposedAgent``) is inherited from ``BOptGPAX``.  See
[faro/agents/bo_oscillation.py](../../faro/agents/bo_oscillation.py).


## Configure and run Bayesian Optimisation

**Parameter space (v11 — same reparameterisation as v9):**

- `light_budget`: 4000–60000 ms (step 1000) = 57 levels
- `ramp_fraction`: 0.0–1.0 (step 0.1) = 11 levels
- `pulse_interval`: 1–10 frames step 2 = 5 levels ⇒ {1, 3, 5, 7, 9}
- Total grid: 57 × 11 × 5 = **3135 candidate conditions**

**Initial exploration cap (`initial_exploration_cap=0.7`):**
First 2 phases (farthest-point sampling) restrict to `light_budget ≤
43200 ms`, `ramp_fraction ≤ 0.7`, `pulse_interval ≤ 7`.

**Covariates:** `n_cells`, `optortk_expression` (log-scaled),
`baseline_cnr`. Marginalised over during acquisition.

**Default objective: `frac_responders`** — per-FOV fraction of cells
whose per-cell mean classifier probability across the scoring window is
`≥ 0.75`.  Same as v9.

**FOV finder pre-screen.**  Each candidate FOV is now imaged with
miRFP + mScarlet3, segmented, run through `FE_ErkKtr` for per-cell CNR,
and rejected unless ≥ 75 % of cells have CNR < 1.0.  See the
`FOVFinderAgent(...)` invocation below.


In [ ]:
from faro.agents import FOVFinderAgent, ComposedAgent, FOVCondition

# ---------------------------------------------------------------------------
# v11 reparameterisation: light_budget / ramp_fraction / pulse_interval
# ---------------------------------------------------------------------------
# Identical to v9's BudgetOscillationBO except: pulse_interval grid widened
# (1-10 step 2 = {1, 3, 5, 7, 9}).  The class is copied here unchanged so
# the notebook stays self-contained for post-hoc replay.


class BudgetOscillationBO(OscillationBO):
    """OscillationBO subclass that reparameterises the BO search space.

    The BO sees (light_budget, ramp_fraction, pulse_interval) instead of
    (stim_exposure, ramp, pulse_interval).  Conversion to per-pulse base
    and ramp is done in :meth:`_create_events_for_batch` before delegating
    to the parent.  ``df_results`` ends up with all five columns:
    ``light_budget``, ``ramp_fraction`` (BO axes) plus ``stim_exposure``,
    ``ramp``, ``pulse_interval`` (physical / derived).  Extra per-FOV
    AUC columns are added for the ``auc_per_light`` objective.
    """

    @staticmethod
    def _budget_to_base_ramp(budget, ramp_fraction, n_pulses):
        if n_pulses <= 0:
            return 0.0, 0.0
        if n_pulses == 1:
            return float(budget), 0.0
        base = float(budget) * (1.0 - float(ramp_fraction)) / n_pulses
        ramp = 2.0 * float(budget) * float(ramp_fraction) / (n_pulses * (n_pulses - 1))
        return base, ramp

    def _create_events_for_batch(self, param_list):
        converted = []
        for params in param_list:
            pi = max(1, int(params["pulse_interval"]))
            n = len(range(self.first_frame_stim, self.last_frame_stim, pi))
            base, ramp = self._budget_to_base_ramp(
                params["light_budget"], params["ramp_fraction"], n
            )
            converted.append(
                {
                    **params,
                    "stim_exposure": base,
                    "ramp": ramp,
                    "pulse_interval": pi,
                }
            )
            print(
                f"    [budget reparam] light_budget={params['light_budget']:.0f}ms, "
                f"ramp_fraction={params['ramp_fraction']:.2f}, "
                f"pulse_interval={pi}: n_pulses={n} -> "
                f"base={base:.1f}ms, ramp={ramp:.2f}ms/pulse, "
                f"peak={base + ramp * (n - 1):.1f}ms"
            )
        return super()._create_events_for_batch(converted)

    def _compute_auc_per_fov(self, fov_tracks):
        """Per-FOV AUC-above-baseline metrics for the auc_per_light objective.

        For each valid cell:
            baseline_cell = mean(CNR_t for t in baseline frames)
            auc_cell      = Σ (CNR_t - baseline_cell)
                                over t ∈ [first_frame_stim, n_frames)
                                (stim + recovery window)
            auc_cell      = max(0, auc_cell)              # clip per cell

        Per-FOV value:
            auc_above_baseline = median over valid cells of auc_cell

        Returns the per-FOV AUC plus diagnostic stim-window means used
        by downstream plots (`mean_cnr_stim`, `mean_cnr_above_baseline`).
        """
        phase_id = self._current_phase_id
        out = {}
        min_frames = int(self.min_track_fraction * self.n_frames)

        for fov_idx, df_tracks in fov_tracks.items():
            if df_tracks.empty or "particle" not in df_tracks.columns:
                continue
            if "phase_id" in df_tracks.columns:
                df_phase = df_tracks[df_tracks["phase_id"] == phase_id]
            else:
                df_phase = df_tracks
            if df_phase.empty:
                continue

            cnr_col = (
                "cnr"
                if "cnr" in df_phase.columns
                else "cnr_median" if "cnr_median" in df_phase.columns else None
            )
            if cnr_col is None:
                continue

            frames_per_cell = df_phase.groupby("particle")["fov_timestep"].nunique()
            well_tracked = set(frames_per_cell[frames_per_cell >= min_frames].index)

            baseline_df = df_phase[df_phase["fov_timestep"] < self.n_baseline_frames]
            per_cell_baseline = (
                baseline_df.groupby("particle")[cnr_col].mean().dropna()
                if not baseline_df.empty
                else pd.Series(dtype=float)
            )

            valid = well_tracked
            if self.max_baseline_cnr is not None and len(per_cell_baseline) > 0:
                valid = valid & set(
                    per_cell_baseline[per_cell_baseline < self.max_baseline_cnr].index
                )
            if not valid:
                continue

            # AUC integration window: stim + recovery (everything after baseline).
            post_bl = df_phase[
                (df_phase["fov_timestep"] >= self.first_frame_stim)
                & (df_phase["fov_timestep"] < self.n_frames)
                & (df_phase["particle"].isin(valid))
            ]
            if post_bl.empty:
                continue

            # Per-cell AUC = Σ (CNR_t - baseline_cell) over the window.
            # Implemented as sum(CNR) - baseline_cell * n_frames_observed
            # so cells with frame gaps don't get an inflated baseline credit.
            per_cell_sum = post_bl.groupby("particle")[cnr_col].sum()
            per_cell_n = post_bl.groupby("particle")[cnr_col].count()
            per_cell_baseline_aligned = per_cell_baseline.reindex(
                per_cell_sum.index
            ).fillna(0.0)
            per_cell_auc = per_cell_sum - per_cell_baseline_aligned * per_cell_n
            per_cell_auc_clipped = per_cell_auc.clip(lower=0.0)

            # Diagnostic: stim-window means (kept so the visualisation cells
            # that reference these columns still work).
            in_stim = df_phase[
                (df_phase["fov_timestep"] >= self.first_frame_stim)
                & (df_phase["fov_timestep"] < self.last_frame_stim)
                & (df_phase["particle"].isin(valid))
            ]
            if in_stim.empty:
                mean_cnr_stim = 0.0
                mean_cnr_above_baseline = 0.0
            else:
                per_cell_mean_cnr_stim = in_stim.groupby("particle")[cnr_col].mean()
                per_cell_above_baseline = (
                    per_cell_mean_cnr_stim
                    - per_cell_baseline.reindex(per_cell_mean_cnr_stim.index).fillna(
                        0.0
                    )
                )
                mean_cnr_stim = float(per_cell_mean_cnr_stim.mean())
                mean_cnr_above_baseline = float(per_cell_above_baseline.mean())

            out[int(fov_idx)] = {
                "auc_above_baseline": float(per_cell_auc_clipped.median()),
                "n_valid_cells_auc": int(len(per_cell_auc_clipped)),
                "mean_cnr_stim": mean_cnr_stim,
                "mean_cnr_above_baseline": mean_cnr_above_baseline,
            }
        return out

    def _preprocess_results(self, fov_tracks):
        df = super()._preprocess_results(fov_tracks)
        if df.empty:
            return df

        budgets = []
        ramp_fracs = []
        for fov_idx in df["fov"].astype(int):
            params = self._current_condition_map.get(int(fov_idx), {})
            budgets.append(float(params.get("light_budget", float("nan"))))
            ramp_fracs.append(float(params.get("ramp_fraction", float("nan"))))
        df["light_budget"] = budgets
        df["ramp_fraction"] = ramp_fracs

        auc_by_fov = self._compute_auc_per_fov(fov_tracks)
        df["mean_cnr_stim"] = [
            auc_by_fov.get(int(f), {}).get("mean_cnr_stim", 0.0)
            for f in df["fov"].astype(int)
        ]
        df["mean_cnr_above_baseline"] = [
            auc_by_fov.get(int(f), {}).get("mean_cnr_above_baseline", 0.0)
            for f in df["fov"].astype(int)
        ]
        df["auc_above_baseline"] = [
            auc_by_fov.get(int(f), {}).get("auc_above_baseline", 0.0)
            for f in df["fov"].astype(int)
        ]
        df["n_valid_cells_auc"] = [
            auc_by_fov.get(int(f), {}).get("n_valid_cells_auc", 0)
            for f in df["fov"].astype(int)
        ]
        light_seconds = (df["light_budget"].clip(lower=1.0)) / 1000.0
        # auc_per_light: per-FOV median per-cell AUC over [first_frame_stim,
        # n_frames), clipped at 0 per cell, divided by total light dose in s.
        df["auc_per_light"] = df["auc_above_baseline"] / light_seconds
        return df


# --- BO parameters ----------------------------------------------------------
# Order matters: x_total_linespace is built via np.meshgrid(..., indexing="ij")
# so the FIRST parameter varies SLOWEST in the linearised grid.  The plot
# code below relies on (light_budget, ramp_fraction, pulse_interval) order
# when slicing acq_values_total at fixed pulse_interval.
bo_params = [
    BO_Parameter(name="light_budget", bounds=(4000.0, 60000.0), spacing=1000.0),
    BO_Parameter(name="ramp_fraction", bounds=(0.0, 1.0), spacing=0.1),
    BO_Parameter(
        name="pulse_interval",
        bounds=(1.0, 10.0),  # v11: was (1, 5) in v9; widened.
        spacing=2.0,  # v11: was 1.0 in v9 -> {1, 3, 5, 7, 9}.
        param_type="int",
    ),
]

# --- Covariates (observed, not controlled) ---
bo_covariates = [
    BO_Covariate(name="n_cells"),
    BO_Covariate(name="optortk_expression", log_scale=True),
    BO_Covariate(name="baseline_cnr"),
]

# --- Objective ---
# v12: switched from frac_responders to auc_per_light. With light_budget
# kept as a BO axis (4000-60000 ms), this rewards patterns that produce
# the biggest CNR-above-baseline per millisecond of LED-on time -- so the
# BO is free to drive the dose DOWN if a low-budget pattern is efficient.
bo_objective = BO_Objective(name="auc_per_light", goal="maximize")

# --- Pre-phase agent: FOV finder with CNR pre-screen --------------------
# v11: FOV finder now images BOTH miRFP and mScarlet3 (so FE_ErkKtr can
# compute per-cell CNR on the candidate frame) and rejects FOVs where
# fewer than 75 % of cells have baseline CNR < 1.0.  Filters out fields
# the per-cell `max_baseline_cnr=1.0` gate downstream would later strip
# down to <5 valid cells, before any wet-lab time is spent on them.
fov_finder = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=WELLS,
    wells_per_phase=N_WELLS_PER_PHASE,
    fovs_per_well=FOVS_PER_WELL,
    n_candidates_per_well=FOV_N_CANDIDATES_PER_WELL,
    border_um=FOV_BORDER_UM,
    min_distance_um=FOV_MIN_DISTANCE_UM,
    min_cells=FOV_MIN_CELLS,
    max_cells=250,
    # v11: acquire BOTH channels so FE_ErkKtr (which reads channel index 1)
    # has the mScarlet3 (ERK-KTR) signal it needs to compute per-cell CNR.
    # Doubles the per-candidate scan time vs v9 but lets us screen on
    # biology, not just count.
    imaging_channels=imaging_channels,
    segmentator=segmentator,
    seg_channel_index=0,
    feature_extractor=FE_ErkKtr("labels"),
    fov_conditions=[
        FOVCondition("cnr", "below", 1.0, min_fraction=0.75),
    ],
    random_seed=None,
    selection_mode="extremes",
    cycle_wells=True,
    verbose=False,
    z=None,
)
print(
    f"FOVFinder: {len(WELLS)} wells queued -> "
    f"{fov_finder.n_remaining_phases} phases @ {N_WELLS_PER_PHASE} wells/phase"
    f"  (cycle_wells=True: refills indefinitely)"
)
print("  Pre-screen: FE_ErkKtr + FOVCondition(cnr<1.0, min_fraction=0.75)")

# --- BO agent ---
agent = BudgetOscillationBO(
    storage_path=path,
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    imaging_channels=imaging_channels,
    stim_channel=stim_channel,
    optocheck_channel=optocheck_channel,
    osc_clf=osc_clf,
    osc_scaler=osc_scaler,
    osc_feature_cols=osc_feature_cols,
    osc_cfg=osc_cfg,
    osc_predict_fn=predict_trace,
    min_osc_probability=MIN_OSC_PROBABILITY,
    min_consecutive_windows=MIN_CONSECUTIVE_WINDOWS,
    min_fft_amplitude=MIN_FFT_AMPLITUDE,
    n_baseline_frames=N_FRAMES_BASELINE,
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=N_PHASES,
    n_conditions_per_iter=N_CONDITIONS_PER_ITER,
    n_initial_phases=2,
    initial_exploration_cap=0.7,
    frac_responder_threshold=0.75,
    acquisition_function="ei",
    n_cov_samples=16,  # was 40 in v9 — dropped to 16 to halve robust-acq predict cost.
    ei_xi=0.1,
    ei_xi_final=0.01,
    ei_num_samples=4,  # was 8 in v9 — dropped to 4 (still 4*num_mcmc=3200 effective draws).
    ei_xi_decay_fraction=0.7,
    use_closed_form_predict=True,  # ~10x faster acquisition; see BOptGPAX._predict_mean_diag_var_closed_form for caveats.
    verbose=True,
    max_baseline_cnr=1.00,
    min_track_fraction=0.8,
)

# --- Composed agent: drives FOV finder + BO for N_PHASES phases ---
composed_agent = ComposedAgent(
    inner_agent=agent,
    pre_phase_agents=[fov_finder],
    n_phases=N_PHASES,
)

# --- Create controller ---
ctrl = Controller(mic, pipeline, writer=writer, agent=composed_agent)

print(f"Parameter grid: {len(agent.x_total_linespace)} candidate conditions")
print(
    f"Phases: {N_PHASES}  ({agent.n_initial_phases} initial-spread + "
    f"{N_PHASES - agent.n_initial_phases} BO batches)"
)
print(
    f"Conditions per phase: {N_CONDITIONS_PER_ITER}  "
    f"FOVs per condition: {FOVS_PER_CONDITION}  Total FOVs/phase: {N_FOVS}"
)
print(f"Total FOV observations after {N_PHASES} phases: ~{N_PHASES * N_FOVS}")
print(f"BO objective: {bo_objective.name} (goal={bo_objective.goal})")

In [ ]:
# ---------------------------------------------------------------------------
# 3D-aware live plot override (v9: light_budget × ramp_fraction × pulse_interval)
# ---------------------------------------------------------------------------
# OscillationBO._plot_live and _plot_landscape_and_acq_from_context build a
# 2D ctrl_grid hardcoded for 2 control parameters and label the axes
# stim_exposure / ramp.  v9 has 3 BO axes (light_budget, ramp_fraction,
# pulse_interval) so we still want a panel grid sliced at each
# pulse_interval level, but on the new (budget × fraction) plane.
#
# We rebind _plot_live on this agent instance (no library edit) so each
# phase plots a 3-row × N-pulse_interval panel:
#   row 0 — measured objective per FOV (scatter)
#   row 1 — GP-predicted landscape (light_budget × ramp_fraction at fixed pulse_interval)
#   row 2 — acquisition surface (same slicing)
import types
import matplotlib.pyplot as _plt

# Axis label helper: nice labels for the v9 BO axes; raw name fallback.
_AXIS_LABELS = {
    "light_budget": "light_budget (ms)",
    "ramp_fraction": "ramp_fraction (0=flat, 1=full ramp)",
    "pulse_interval": "pulse_interval (frames)",
    "stim_exposure": "stim_exposure (ms)",
    "ramp": "ramp (ms/pulse)",
}


def _axis_label(name):
    return _AXIS_LABELS.get(name, name)


def _format_level(name, level):
    return f"{int(level)}" if name == "pulse_interval" else f"{float(level):.2f}"


def _level_mask(arr, name, level):
    if name == "pulse_interval":
        return np.round(np.asarray(arr)).astype(int) == int(level)
    return np.isclose(np.asarray(arr).astype(float), float(level))


def _plot_landscape_and_acq_3d(
    self,
    ctx,
    df_results,
    ax_mean,
    ax_acq,
    fig,
    *,
    third_param_level,
    third_param_name,
    p1_name,
    p2_name,
):
    gp_model = ctx["gp_model"]
    x_scaler = ctx["x_scaler"]
    y_scaler = ctx["y_scaler"]
    rng_key_predict = ctx["rng_key_predict"]
    acq_values_total = ctx["acq_values_total"]
    acquisition_used = ctx["acquisition_used"]
    x_unmeasured = ctx["x_unmeasured_at_computation"]

    x_total_ctrl = self.x_total_linespace.copy()
    unique_x1 = np.unique(x_total_ctrl[:, 0])
    unique_x2 = np.unique(x_total_ctrl[:, 1])
    n_ctrl = len(unique_x1) * len(unique_x2)

    # Build the 3D ctrl_grid with the 3rd parameter pinned at this level.
    # Order matches np.meshgrid(..., indexing="ij") + flatten on the BO
    # parameter grid, so reshape(len(unique_x1), len(unique_x2)) is valid.
    ctrl_grid = np.array(
        [[x1, x2, third_param_level] for x1 in unique_x1 for x2 in unique_x2]
    )

    if len(self.bo_covariates) > 0 and not df_results.empty:
        cov_cols = [c.name for c in self.bo_covariates]
        cov_vals_full = np.asarray(df_results[cov_cols].to_numpy(), dtype=float)
        n_cov_samples = 50
        _plot_rng = np.random.default_rng(0)
        row_idx = _plot_rng.integers(0, cov_vals_full.shape[0], size=n_cov_samples)
        cov_samples_joint = cov_vals_full[row_idx]
    else:
        n_cov_samples = 1
        cov_samples_joint = None

    if cov_samples_joint is not None:
        x_grid_full = np.hstack(
            [
                np.repeat(ctrl_grid, n_cov_samples, axis=0),
                np.tile(cov_samples_joint, (n_ctrl, 1)),
            ]
        )
    else:
        x_grid_full = ctrl_grid

    x_grid_scaled = x_scaler.transform(x_grid_full)

    from faro.agents.bo_optimization_sparse import _safe_batch_size

    n_rows = np.asarray(x_grid_scaled).shape[0]
    _bs = _safe_batch_size(n_rows, 1000)
    y_pred_scaled, _ = gp_model.predict_in_batches(
        rng_key_predict,
        x_grid_scaled,
        batch_size=_bs,
        n=self.ei_num_samples,
        noiseless=True,
    )
    y_pred = y_scaler.inverse_transform(
        np.asarray(y_pred_scaled).reshape(-1, 1)
    ).flatten()

    if len(self.bo_covariates) > 0:
        y_pred_marg = y_pred.reshape(n_ctrl, n_cov_samples).mean(axis=1)
    else:
        y_pred_marg = y_pred

    X_mesh, Y_mesh = np.meshgrid(unique_x1, unique_x2, indexing="ij")
    y_pred_2d = y_pred_marg.reshape(len(unique_x1), len(unique_x2))

    level_label = (
        f"\n{third_param_name}={_format_level(third_param_name, third_param_level)}"
    )
    overlay_df = df_results[
        _level_mask(
            df_results[third_param_name].values, third_param_name, third_param_level
        )
    ]

    im1 = ax_mean.pcolormesh(X_mesh, Y_mesh, y_pred_2d, cmap="viridis", shading="auto")
    fig.colorbar(im1, ax=ax_mean, label=f"predicted {self.objective_metric.name}")
    if not overlay_df.empty:
        ax_mean.scatter(
            overlay_df[p1_name],
            overlay_df[p2_name],
            c="white",
            s=15,
            alpha=0.6,
            marker="x",
            linewidths=0.8,
        )
    ax_mean.set_xlabel(_axis_label(p1_name))
    ax_mean.set_ylabel(_axis_label(p2_name))
    ax_mean.set_title(f"GP predicted landscape{level_label}")

    # Pad acq_values_total to full x_total_linespace ordering, then slice
    # to rows where the third param matches and reshape to (x1, x2).
    acq_marg = np.asarray(acq_values_total)
    full_n = len(x_total_ctrl)
    if len(acq_marg) != full_n:
        acq_full = np.zeros(full_n)
        for j, pt in enumerate(x_unmeasured):
            diffs = np.abs(x_total_ctrl - pt).sum(axis=1)
            idx = np.argmin(diffs)
            if j < len(acq_marg):
                acq_full[idx] = float(acq_marg[j])
        acq_marg = acq_full

    mask = _level_mask(x_total_ctrl[:, 2], third_param_name, third_param_level)
    acq_2d = acq_marg[mask].reshape(len(unique_x1), len(unique_x2))

    im2 = ax_acq.pcolormesh(X_mesh, Y_mesh, acq_2d, cmap="inferno", shading="auto")
    acq_label = acquisition_used.upper()
    fig.colorbar(im2, ax=ax_acq, label=f"{acq_label} acquisition")
    if not overlay_df.empty:
        ax_acq.scatter(
            overlay_df[p1_name],
            overlay_df[p2_name],
            c="white",
            s=15,
            alpha=0.6,
            marker="x",
            linewidths=0.8,
        )

    picks = getattr(self, "_current_batch_picks", None)
    if picks:
        picks_arr = np.array(picks, dtype=float)
        if picks_arr.shape[1] >= 3:
            level_mask_arr = _level_mask(
                picks_arr[:, 2], third_param_name, third_param_level
            )
            picks_arr = picks_arr[level_mask_arr]
        if len(picks_arr) > 0:
            ax_acq.scatter(
                picks_arr[:, 0],
                picks_arr[:, 1],
                c="cyan",
                s=120,
                marker="X",
                edgecolors="k",
                linewidths=1.0,
                zorder=10,
                label="next conditions",
            )
            ax_acq.legend(loc="upper right", fontsize=7)

    ax_acq.set_xlabel(_axis_label(p1_name))
    ax_acq.set_ylabel(_axis_label(p2_name))
    ax_acq.set_title(f"Acquisition {acq_label}{level_label}")


def _plot_live_3d(self, df_results, iteration_label, save_subdir="after"):
    p1_name = self.parameters_to_optimize[0].name
    p2_name = self.parameters_to_optimize[1].name
    third_param_name = self.parameters_to_optimize[2].name
    levels = sorted(np.unique(self.x_total_linespace[:, 2]).tolist())
    n_cols = len(levels)

    fig, axes = _plt.subplots(
        3, n_cols, figsize=(6 * n_cols, 14), dpi=200, squeeze=False
    )
    fig.suptitle(iteration_label, fontsize=13, fontweight="bold")

    obj_name = self.objective_metric.name
    ctx = getattr(self, "_last_plot_context", None)

    # Per-axis jitter sized by the parameter grid spacing so samples in
    # the per-FOV scatter don't overlap.
    p1_step = float(self.parameters_to_optimize[0].spacing or 1.0)
    p2_step = float(self.parameters_to_optimize[1].spacing or 1.0)

    for col, level in enumerate(levels):
        ax_meas = axes[0, col]
        ax_landscape = axes[1, col]
        ax_acq = axes[2, col]

        slice_mask = _level_mask(
            df_results[third_param_name].values, third_param_name, level
        )
        df_slice = df_results[slice_mask]
        level_label = f"\n{third_param_name}={_format_level(third_param_name, level)}"

        if not df_slice.empty:
            jitter_x = self._rng.normal(0, p1_step * 0.15, size=len(df_slice))
            jitter_y = self._rng.normal(0, p2_step * 0.15, size=len(df_slice))
            sc = ax_meas.scatter(
                df_slice[p1_name].values + jitter_x,
                df_slice[p2_name].values + jitter_y,
                c=df_slice[obj_name],
                cmap="viridis",
                s=30,
                edgecolors="k",
                linewidths=0.3,
                alpha=0.8,
            )
            fig.colorbar(sc, ax=ax_meas, label=obj_name)
        ax_meas.set_xlabel(_axis_label(p1_name))
        ax_meas.set_ylabel(_axis_label(p2_name))
        ax_meas.set_title(f"Measured {obj_name}{level_label}")

        if ctx is None:
            for ax in (ax_landscape, ax_acq):
                ax.text(
                    0.5,
                    0.5,
                    "GP not fit yet\n(initial batch)",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=11,
                    color="gray",
                )
                ax.set_xlabel(_axis_label(p1_name))
                ax.set_ylabel(_axis_label(p2_name))
            ax_landscape.set_title(f"GP predicted landscape{level_label}")
            ax_acq.set_title(f"Acquisition{level_label}")
        else:
            try:
                _plot_landscape_and_acq_3d(
                    self,
                    ctx,
                    df_results,
                    ax_landscape,
                    ax_acq,
                    fig,
                    third_param_level=level,
                    third_param_name=third_param_name,
                    p1_name=p1_name,
                    p2_name=p2_name,
                )
            except Exception as e:
                for ax in (ax_landscape, ax_acq):
                    ax.text(
                        0.5,
                        0.5,
                        f"Plot failed:\n{type(e).__name__}: {e}",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                        fontsize=9,
                        color="red",
                    )

    _plt.tight_layout()

    plots_dir = os.path.join(self.storage_path, "plots", save_subdir)
    os.makedirs(plots_dir, exist_ok=True)
    fig.savefig(
        os.path.join(plots_dir, f"phase_{self._current_phase_id:03d}.png"),
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        os.path.join(plots_dir, f"phase_{self._current_phase_id:03d}.svg"),
        bbox_inches="tight",
    )
    _plt.show()


# Bind as a method on the agent instance (does NOT touch the library class).
agent._plot_live = types.MethodType(_plot_live_3d, agent)
print(
    f"Live-plot override installed: 3-row × "
    f"{len(np.unique(agent.x_total_linespace[:, 2]))}-{agent.parameters_to_optimize[2].name} panel grid."
)

In [ ]:
# --- Run the composed (FOV finder + BO) loop ---
# Each phase: FOVFinder picks 18 fresh FOVs (3 FOVs in 6 fresh wells), then
# OscillationBO runs one batch BO iteration over those FOVs.  The GP
# accumulates observations across all phases.


# Disconnect napari live view so it does not interfere with the acquisition engine
try:
    mm_wdg._core_link.cleanup()
except:
    pass

sleep_time = 3600 * 10  # sleep until tomorrow morning
for t in range(0, sleep_time):
    time.sleep(1)


composed_agent.run()

In [ ]:
# Post-processing: merge per-FOV tracks
utils.generate_exp_data_from_tracks(path)

## Visualise results

In [ ]:
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results
    obj_name = agent.objective_metric.name

    # Per-pulse_interval panel grid:
    #   row 0 -- per-FOV scatter (light_budget × ramp_fraction colored by objective)
    #   row 1 -- per-condition mean across FOVs (size = n_fovs in that condition)
    pi_levels = sorted(df["pulse_interval"].astype(int).unique().tolist())
    n_levels = len(pi_levels)

    # Shared colour scale across panels so colours are comparable.
    vmin = float(df[obj_name].min())
    vmax = float(df[obj_name].max())

    fig, axes = plt.subplots(
        2, n_levels, figsize=(5 * n_levels, 10), squeeze=False, sharex=True, sharey=True
    )

    for col, pi in enumerate(pi_levels):
        df_pi = df[df["pulse_interval"].astype(int) == pi]

        # --- row 0: per-FOV scatter ---
        ax = axes[0, col]
        if not df_pi.empty:
            sc = ax.scatter(
                df_pi["light_budget"],
                df_pi["ramp_fraction"],
                c=df_pi[obj_name],
                cmap="viridis",
                s=40,
                edgecolors="k",
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
            )
            fig.colorbar(sc, ax=ax, label=obj_name)
        ax.set_xlabel("light_budget (ms)")
        if col == 0:
            ax.set_ylabel("ramp_fraction")
        ax.set_title(f"Per-FOV  |  pulse_interval={pi}\n(n_FOVs={len(df_pi)})")

        # --- row 1: per-condition aggregation ---
        ax = axes[1, col]
        cond_agg = (
            df_pi.groupby(["light_budget", "ramp_fraction"])
            .agg(
                mean_obj=(obj_name, "mean"),
                n_fovs=(obj_name, "count"),
            )
            .reset_index()
        )
        if not cond_agg.empty:
            sc = ax.scatter(
                cond_agg["light_budget"],
                cond_agg["ramp_fraction"],
                c=cond_agg["mean_obj"],
                s=cond_agg["n_fovs"] * 30,
                cmap="viridis",
                edgecolors="k",
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
            )
            fig.colorbar(sc, ax=ax, label=f"mean {obj_name}")
        ax.set_xlabel("light_budget (ms)")
        if col == 0:
            ax.set_ylabel("ramp_fraction")
        ax.set_title(
            f"Per-condition mean  |  pulse_interval={pi}\n(marker size = n_FOVs)"
        )

    fig.suptitle(
        f"Measured {obj_name} per pulse_interval level\n"
        "(BO axes: light_budget × ramp_fraction)",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # --- Convergence (params-agnostic, single panel) ---
    if agent._iteration_means:
        fig, ax = plt.subplots(figsize=(8, 5))
        means = np.array(agent._iteration_means)
        cumulative_best = np.maximum.accumulate(means)
        ax.plot(means, "o-", alpha=0.5, label="iteration mean")
        ax.plot(cumulative_best, "s-", color="red", label="cumulative best")
        ax.set_xlabel("Iteration")
        ax.set_ylabel(obj_name)
        ax.set_title("BO convergence")
        ax.legend()
        plt.tight_layout()
        plt.show()

    # --- Budget vs response trade-off (the v9 question) ---
    # For each condition (mean across FOVs), plot total light vs raw
    # response.  Points near the upper-left are the most light-efficient
    # patterns: high response at low total light.  Colour by ramp_fraction,
    # marker shape by pulse_interval.
    fig, ax = plt.subplots(figsize=(9, 6))
    pi_markers = {1: "o", 2: "s", 3: "^", 4: "D", 5: "v"}
    cond_full = (
        df.groupby(["light_budget", "ramp_fraction", "pulse_interval"])
        .agg(
            mean_response=("mean_cnr_above_baseline", "mean"),
            mean_efficiency=("auc_per_light", "mean"),
            n_fovs=("auc_per_light", "count"),
        )
        .reset_index()
    )
    for pi, marker in pi_markers.items():
        sub = cond_full[cond_full["pulse_interval"] == pi]
        if sub.empty:
            continue
        sc = ax.scatter(
            sub["light_budget"],
            sub["mean_response"],
            c=sub["ramp_fraction"],
            cmap="plasma",
            s=sub["n_fovs"] * 25,
            marker=marker,
            edgecolors="k",
            linewidths=0.5,
            vmin=0.0,
            vmax=1.0,
            label=f"pulse_interval={pi}",
        )
    cb = fig.colorbar(sc, ax=ax, label="ramp_fraction")
    ax.set_xlabel("light_budget (ms)")
    ax.set_ylabel("mean_cnr_above_baseline (per-FOV mean)")
    ax.set_title(
        "Budget vs response trade-off\n"
        "(upper-left = most light-efficient; marker size = n_FOVs)"
    )
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    # --- Best condition by configured BO objective ---
    cond_agg_all = (
        df.groupby(["light_budget", "ramp_fraction", "pulse_interval"])
        .agg(
            mean_obj=(obj_name, "mean"),
            mean_response=("mean_cnr_above_baseline", "mean"),
            mean_efficiency=("auc_per_light", "mean"),
            mean_frac_resp=("frac_responders", "mean"),
            n_fovs=(obj_name, "count"),
            mean_base_exposure=("stim_exposure", "mean"),
            mean_ramp=("ramp", "mean"),
        )
        .reset_index()
    )
    best_idx = cond_agg_all["mean_obj"].idxmax()
    best = cond_agg_all.loc[best_idx]
    print(f"\nBest condition by {obj_name} (mean across FOVs):")
    print(f"  light_budget          = {best['light_budget']:.0f} ms")
    print(f"  ramp_fraction         = {best['ramp_fraction']:.2f}")
    print(f"  pulse_interval        = {int(best['pulse_interval'])} frames")
    print(f"  → derived base_exp    = {best['mean_base_exposure']:.1f} ms")
    print(f"  → derived ramp        = {best['mean_ramp']:.2f} ms/pulse")
    print(f"  mean {obj_name:<17}= {best['mean_obj']:.4f}")
    print(f"  mean_cnr_above_baseline = {best['mean_response']:.4f}")
    print(f"  auc_per_light         = {best['mean_efficiency']:.4f}")
    print(f"  frac_responders       = {best['mean_frac_resp']:.4f}")
    print(f"  tested on {int(best['n_fovs'])} FOVs")
else:
    print("No data collected yet.")

In [ ]:
# GP-predicted landscape, marginalised over covariates the SAME way the BO
# acquisition does: average GP predictions over joint samples from the
# empirical covariate distribution.  Rendered as one (light_budget x
# ramp_fraction) heatmap per pulse_interval level.
import gpax
import gpax.utils
from faro.agents.bo_optimization_sparse import _safe_batch_size

if agent.model is not None and agent.x is not None:
    obj_name = agent.objective_metric.name
    rng_key, rng_key_pred = gpax.utils.get_keys()

    budget_vals = np.arange(4000.0, 60000.0 + 1000.0, 1000.0)  # 57 levels
    rf_vals = np.arange(0.0, 1.0 + 0.1, 0.1)  # 11 levels
    pi_levels = sorted(np.unique(agent.x_total_linespace[:, 2]).astype(int).tolist())
    n_pi = len(pi_levels)
    n_b = len(budget_vals)
    n_r = len(rf_vals)

    df = agent.df_results
    n_cov_samples = 50
    cov_vals_full = df[[c.name for c in agent.bo_covariates]].to_numpy(dtype=float)
    if cov_vals_full.shape[0] == 0:
        raise RuntimeError("No covariate observations available for marginalisation.")
    _rng = np.random.default_rng(0)
    idx = _rng.integers(0, cov_vals_full.shape[0], size=n_cov_samples)
    cov_samples = cov_vals_full[idx]

    ctrl_grid = np.array(
        [[b, r, float(pi)] for pi in pi_levels for b in budget_vals for r in rf_vals]
    )
    n_grid = ctrl_grid.shape[0]

    x_full = np.hstack(
        [
            np.repeat(ctrl_grid, n_cov_samples, axis=0),
            np.tile(cov_samples, (n_grid, 1)),
        ]
    )
    x_full_scaled = agent._x_scaler.transform(x_full)

    n_rows = np.asarray(x_full_scaled).shape[0]
    bs = _safe_batch_size(n_rows, 2000)
    y_pred_scaled, _ = agent.model.predict_in_batches(
        rng_key_pred,
        x_full_scaled,
        batch_size=bs,
        n=1,
        noiseless=True,
    )
    y_pred = agent._y_scaler.inverse_transform(
        np.asarray(y_pred_scaled).reshape(-1, 1)
    ).flatten()

    Y_marg = y_pred.reshape(n_grid, n_cov_samples).mean(axis=1)
    Y_marg_3d = Y_marg.reshape(n_pi, n_b, n_r)

    Y_grids = {pi: Y_marg_3d[i] for i, pi in enumerate(pi_levels)}
    pi_optima = {}
    for i, pi in enumerate(pi_levels):
        flat_idx = int(np.argmax(Y_marg_3d[i]))
        b_idx, r_idx = np.unravel_index(flat_idx, (n_b, n_r))
        pi_optima[pi] = (
            float(budget_vals[b_idx]),
            float(rf_vals[r_idx]),
            float(Y_marg_3d[i, b_idx, r_idx]),
        )

    vmin = float(min(g.min() for g in Y_grids.values()))
    vmax = float(max(g.max() for g in Y_grids.values()))
    global_best_pi = max(pi_levels, key=lambda p: pi_optima[p][2])

    fig, axes = plt.subplots(
        1, len(pi_levels), figsize=(7 * len(pi_levels), 6), squeeze=False, sharey=True
    )

    for col, pi in enumerate(pi_levels):
        ax = axes[0, col]
        Y_grid = Y_grids[pi]
        cf = ax.contourf(
            rf_vals,
            budget_vals,
            Y_grid,
            levels=20,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        fig.colorbar(cf, ax=ax, label=f"predicted {obj_name}")

        df_pi = df[df["pulse_interval"].astype(int) == pi]
        if not df_pi.empty:
            ax.scatter(
                df_pi["ramp_fraction"],
                df_pi["light_budget"],
                c=df_pi[obj_name],
                cmap="viridis",
                s=30,
                edgecolors="white",
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
                zorder=5,
            )

        opt_b, opt_r, opt_val = pi_optima[pi]
        is_global = pi == global_best_pi
        ax.scatter(
            opt_r,
            opt_b,
            c="red",
            s=240 if is_global else 180,
            marker="*",
            edgecolors="black",
            linewidths=2,
            zorder=10,
            label=(
                f"opt: budget={opt_b:.0f}ms, frac={opt_r:.2f}\n"
                f"pred={opt_val:.3f}{'  (global)' if is_global else ''}"
            ),
        )
        ax.set_xlabel("ramp_fraction")
        if col == 0:
            ax.set_ylabel("light_budget (ms)")
        ax.set_title(f"pulse_interval = {pi}")
        ax.legend(loc="lower right", fontsize=7)

    fig.suptitle(
        f"GP-predicted {obj_name} landscape per pulse_interval\n"
        f"(marginalised over {n_cov_samples} joint covariate samples from df_results)",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    print(f"GP-predicted optimum per pulse_interval (marginalised over covariates):")
    for pi in pi_levels:
        opt_b, opt_r, opt_val = pi_optima[pi]
        marker = "  <- global" if pi == global_best_pi else ""
        print(
            f"  pulse_interval={pi}: light_budget={opt_b:>6.0f}ms  "
            f"ramp_fraction={opt_r:>4.2f}  predicted={obj_name}={opt_val:.4f}{marker}"
        )
    bp = pi_optima[global_best_pi]
    print(
        f"\nGlobal optimum: pulse_interval={global_best_pi}, "
        f"light_budget={bp[0]:.0f}ms, ramp_fraction={bp[1]:.2f}, "
        f"predicted {obj_name}={bp[2]:.4f}"
    )

In [ ]:
# Covariate effects on the configured BO objective (per FOV)
if agent.df_results is not None and not agent.df_results.empty:
    df = agent.df_results
    obj_name = agent.objective_metric.name

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(df["n_cells"], df[obj_name], alpha=0.6, s=20)
    axes[0].set_xlabel("n_cells per FOV")
    axes[0].set_ylabel(obj_name)
    axes[0].set_title(f"Cell density vs {obj_name}")

    axes[1].scatter(df["optortk_expression"], df[obj_name], alpha=0.6, s=20)
    axes[1].set_xlabel("optoRTK expression (mCitrine intensity)")
    axes[1].set_ylabel(obj_name)
    axes[1].set_title(f"optoRTK expression vs {obj_name}")

    axes[2].scatter(df["baseline_cnr"], df[obj_name], alpha=0.6, s=20)
    axes[2].set_xlabel("baseline_cnr (pre-stim CNR)")
    axes[2].set_ylabel(obj_name)
    axes[2].set_title(f"Baseline CNR vs {obj_name}")

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import glob, os

# Per-phase delta of conditions tested.  Each checkpoint stores the
# cumulative df_results, so phase N's new conditions = rows in ckpt N
# that aren't in ckpt N-1.  Rendered in BO-axis terms (light_budget,
# ramp_fraction, pulse_interval) since those are what the BO actually
# searches over.
RUN_PATH = path  # storage_path of this notebook's run
ckpts = sorted(
    glob.glob(os.path.join(RUN_PATH, "checkpoints", "bo_results_phase_*.parquet"))
)

prev_len = 0
summary = []
for f in ckpts:
    phase_id = int(os.path.basename(f).split("_")[-1].split(".")[0])
    df = pd.read_parquet(f)
    df_new = df.iloc[prev_len:]
    prev_len = len(df)
    conds = (
        df_new[["light_budget", "ramp_fraction", "pulse_interval"]]
        .drop_duplicates()
        .values.tolist()
    )
    summary.append(
        {
            "phase_id": phase_id,
            "n_fovs": len(df_new),
            "conditions": [
                f"(budget={int(b)}ms, frac={r:.2f}, pi={int(p)})" for b, r, p in conds
            ],
        }
    )

for row in summary:
    print(
        f"Phase {row['phase_id']:2d}  ({row['n_fovs']:2d} FOVs): "
        f"{'  |  '.join(row['conditions'])}"
    )